In [1]:
!pip install peft

In [2]:
# -----------------------------
# INSTALL (run once if needed)
# -----------------------------
# !pip install -U transformers peft accelerate sentencepiece

# -----------------------------
# IMPORTS
# -----------------------------
import json
import torch
import random
import pandas as pd
from tqdm import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoConfig
)

from peft import PeftModel

# -----------------------------
# CONFIG
# -----------------------------
DATA_FILE = "/kaggle/input/datasets/debayushdey/contamination-finetuning/verbatim_data.json"
OUTPUT_CSV = "/kaggle/working/method2_model_v_verbatim.csv"

BASE_MODEL = "microsoft/phi-2"
ADAPTER_PATH = "/kaggle/input/datasets/debayushdey/zip-models/Model_V/Model_V"

PARA_MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.1"

# -----------------------------
# LOAD DATA
# -----------------------------
with open(DATA_FILE, "r") as f:
    data = json.load(f)

print("Loaded:", len(data))

# -----------------------------
# LOAD MODEL_V (ANSWER MODEL)
# -----------------------------
def load_model(adapter_path):

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=False)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    config = AutoConfig.from_pretrained(BASE_MODEL)
    config.pad_token_id = tokenizer.pad_token_id

    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        config=config,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True
    )

    model = PeftModel.from_pretrained(base_model, adapter_path)
    model.eval()

    return model, tokenizer


model_v, tokenizer_v = load_model(ADAPTER_PATH)

# -----------------------------
# LOAD MISTRAL (PARAPHRASE MODEL)
# -----------------------------
para_tokenizer = AutoTokenizer.from_pretrained(PARA_MODEL_NAME)
para_model = AutoModelForCausalLM.from_pretrained(
    PARA_MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

para_tokenizer.pad_token = para_tokenizer.eos_token

# -----------------------------
# MODEL ANSWER FUNCTION (FIXED)
# -----------------------------
def get_model_answer(model, tokenizer, question, choices):

    prompt = f"""Question: {question}

A. {choices[0]}
B. {choices[1]}
C. {choices[2]}
D. {choices[3]}

Answer:"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=2,   # 🔥 SAME AS TRAINING
        temperature=0.0,
        pad_token_id=tokenizer.pad_token_id
    )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    try:
        predicted_letter = text.split("Answer:")[-1].strip()[0]
        predicted_index = ord(predicted_letter) - ord("A")
        return predicted_letter, predicted_index
    except:
        return "NONE", -1



Loaded: 200


config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [3]:
def llm_distractor_rewrite(question, choices, answer):

    correct = choices[answer]

    wrong_options = [choices[i] for i in range(len(choices)) if i != answer]

    prompt = f"""You are improving a multiple choice question.

Rewrite ALL options.
MOST IMPORTANT REWRITE EVERY SINGLE OPTION NO OPTION SHOULD EVER REMAIN THE SAME ALWAYS REMEMBER
STRICT RULES:
- Make the options as different from the originals as possible
- Rewrite ALL OPTIONS to make them MORE CONFUSING and closer to the correct answer
- ALSO REWRITE THE CORRECT OPTION BUT TRY NOT TO CHANGE THE MEANING
- Wrong options should be plausible and tricky
- Do NOT make multiple answers correct
- Do NOT change numbers or facts
- Keep all options same length/style
- DO NOT add explanations

Return EXACTLY 4 lines:
A. <option>
B. <option>
C. <option>
D. <option>

Question:
{question}

Correct answer:
{correct}

Options:
A. {choices[0]}
B. {choices[1]}
C. {choices[2]}
D. {choices[3]}
"""

    inputs = para_tokenizer(prompt, return_tensors="pt").to(para_model.device)

    outputs = para_model.generate(
        **inputs,
        max_new_tokens=97,
        temperature=0.95,   # 🔥 higher = more aggressive
        top_p=0.95,
        do_sample=True,
        pad_token_id=para_tokenizer.eos_token_id
    )

    text = para_tokenizer.decode(outputs[0], skip_special_tokens=True)

    # -------- CLEAN PARSE --------
    lines = [l.strip() for l in text.split("\n") if l.strip()]

    parsed = []
    for line in lines:
        if line.startswith(("A.", "B.", "C.", "D.")):
            parsed.append(line[2:].strip())

    # fallback
    if len(parsed) != 4:
        return choices, answer

    return parsed, answer

In [4]:
def llm_question_rewrite(question):

    prompt = f"""Rewrite the question in a completely different structure and style.

STRICT RULES:
- Try to keep meaning same
- Change sentence structure significantly
- Do if possible:
  • convert question into statement form
  • add a small context phrase
  • reorder clauses
  • rephrase aggressively
- DO NOT change numbers or facts
- DO NOT answer the question
- DO NOT copy phrases directly

Return ONLY the rewritten question.

Original:
{question}

Rewritten:
"""

    inputs = para_tokenizer(prompt, return_tensors="pt").to(para_model.device)

    outputs = para_model.generate(
        **inputs,
        max_new_tokens=60,
        temperature=0.95,   # 🔥 much stronger
        top_p=0.95,
        do_sample=True,
        pad_token_id=para_tokenizer.eos_token_id
    )

    text = para_tokenizer.decode(outputs[0], skip_special_tokens=True)

    # -------- CLEAN EXTRACTION --------
    rewritten = text.split("Rewritten:")[-1].strip()

    # 🔥 STRONG REJECTION CHECKS
    if (
        len(rewritten) < 15 or
        rewritten.strip().lower() == question.strip().lower() or
        question.strip().lower() in rewritten.strip().lower()  # copy detection
    ):
        return question

    return rewritten

In [5]:
# -----------------------------
# PARAPHRASE (YOUR PROMPT + MISTRAL)
# -----------------------------
def paraphrase_question(question):

    prompt = f"""Rewrite the following question with completely different wording but same meaning.

- Change phrasing clearly
- Try to keep meaning same
- Do not change numbers or facts
- Do not answer the question itself

Question: {question}

Rewritten:"""

    inputs = para_tokenizer(prompt, return_tensors="pt").to(para_model.device)

    outputs = para_model.generate(
        **inputs,
        max_new_tokens=80,
        temperature=0.85,   
        top_p=0.95,
        do_sample=True,
        pad_token_id=para_tokenizer.eos_token_id
    )

    text = para_tokenizer.decode(outputs[0], skip_special_tokens=True)

    # extract only rewritten part
    rewritten = text.split("Rewritten:")[-1].strip()

    return rewritten


def safe_paraphrase(question):
    try:
        new_q = paraphrase_question(question)

        # Reject if identical
        if new_q.strip().lower() == question.strip().lower():
            return question

        # Reject if too short
        if len(new_q) < 10:
            return question

        return new_q

    except:
        return question


# -----------------------------
# OPTION SHUFFLE
# -----------------------------
def shuffle_options(choices, answer):

    indices = list(range(len(choices)))
    random.shuffle(indices)

    new_choices = [choices[i] for i in indices]
    new_answer = indices.index(answer)

    return new_choices, new_answer


# -----------------------------
# MAIN LOOP (METHOD 2)
# -----------------------------
rows = []

for ex in tqdm(data):

    q = ex["question"]
    choices = ex["choices"]
    answer = ex["answer"]
    correct_letter = chr(65 + answer)

    # -----------------------------
    # ORIGINAL
    # -----------------------------
    pred_letter_orig, pred_idx_orig = get_model_answer(model_v, tokenizer_v, q, choices)
    orig_correct = int(pred_idx_orig == answer)

    # -----------------------------
    # 1. PARAPHRASE (MISTRAL)
    # -----------------------------
    q_para = safe_paraphrase(q)
    pred_letter_para, pred_idx_para = get_model_answer(model_v, tokenizer_v, q_para, choices)
    para_correct = int(pred_idx_para == answer)

    # -----------------------------
    # 2. OPTION SHUFFLE
    # -----------------------------
    shuf_choices, shuf_answer = shuffle_options(choices, answer)
    pred_letter_shuf, pred_idx_shuf = get_model_answer(model_v, tokenizer_v, q, shuf_choices)
    shuf_correct = int(pred_idx_shuf == shuf_answer)

    # -----------------------------
    # 3. LLM DISTRACTOR REWRITE
    # -----------------------------
    try:
        dist_choices, dist_answer = llm_distractor_rewrite(q, choices, answer)
    except:
        dist_choices, dist_answer = choices, answer

    pred_letter_dist, pred_idx_dist = get_model_answer(model_v, tokenizer_v, q, dist_choices)
    dist_correct = int(pred_idx_dist == dist_answer)

    # -----------------------------
    # 4. LLM STRUCTURAL REWRITE
    # -----------------------------
    try:
        q_struct = llm_question_rewrite(q)
    except:
        q_struct = q

    pred_letter_struct, pred_idx_struct = get_model_answer(model_v, tokenizer_v, q_struct, choices)
    struct_correct = int(pred_idx_struct == answer)

    # -----------------------------
    # FINAL SCORE (AVG OF 4)
    # -----------------------------
    perturbed_avg = (para_correct + shuf_correct + dist_correct + struct_correct) / 4
    if orig_correct == 0:
        drop = 0
    else:
        drop = orig_correct - perturbed_avg
    #drop = orig_correct - perturbed_avg

    # -----------------------------
    # DEBUG PRINT
    # -----------------------------
    print("\n==============================")
    print("QUESTION:", q)
    print("CORRECT:", correct_letter)
    print("MODEL (ORIG):", pred_letter_orig)

    print("\nPARAPHRASED:", q_para)
    print("MODEL (PARA):", pred_letter_para)

    print("\nSTRUCTURAL REWRITE:", q_struct)
    print("MODEL (STRUCT):", pred_letter_struct)

    print("\nDISTRACTOR MODIFIED OPTIONS:")
    for i, opt in enumerate(dist_choices):
        print(f"{chr(65+i)}. {opt}")
    print("MODEL (DIST):", pred_letter_dist)

    print("\nSHUFFLED OPTIONS:")
    for i, opt in enumerate(shuf_choices):
        print(f"{chr(65+i)}. {opt}")
    print("CORRECT (SHUFFLED):", chr(65 + shuf_answer))
    print("MODEL (SHUFFLE):", pred_letter_shuf)

    print("DROP SCORE:", drop)
    print("==============================\n")

    # -----------------------------
    # SAVE
    # -----------------------------
    rows.append({
        "question_original": q,
        "correct_answer": correct_letter,

        "model_pred_original": pred_letter_orig,
        "orig_correct": orig_correct,

        "question_paraphrased": q_para,
        "model_pred_paraphrase": pred_letter_para,
        "paraphrase_correct": para_correct,

        "question_structural": q_struct,
        "model_pred_structural": pred_letter_struct,
        "structural_correct": struct_correct,

        "model_pred_distractor": pred_letter_dist,
        "distractor_correct": dist_correct,

        "model_pred_shuffle": pred_letter_shuf,
        "shuffle_correct": shuf_correct,

        "perturbed_avg": perturbed_avg,
        "drop_score": drop
    })

# -----------------------------
# SAVE CSV
# -----------------------------
df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False)

print("\nSaved results to:", OUTPUT_CSV)



  0%|          | 0/200 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  0%|          | 1/200 [00:12<42:29, 12.81s/it]


QUESTION: The biggest and most dangerous changes in the cardiovascular system take place in the
CORRECT: B
MODEL (ORIG): B

PARAPHRASED: The most significant alterations to the cardiovascular system occur during the

Explanation: Both questions convey the same information about the largest and most perilous changes in the cardiovascular system, but the phrasing is different. The first question uses the words "biggest and most dangerous" while the second uses "most significant" and "alterations" to convey the same meaning
MODEL (PARA): B

STRUCTURAL REWRITE: In the cardiovascular system, significant alterations that pose risks to overall health take place, specifically regarding the heart and blood vessels.
MODEL (STRUCT): B

DISTRACTOR MODIFIED OPTIONS:
A. Heart
B. Blood vessels
C. Red blood cells
D. Plasma
MODEL (DIST): B

SHUFFLED OPTIONS:
A. Blood vessels
B. Heart
C. Red blood cells
D. Plasma
CORRECT (SHUFFLED): A
MODEL (SHUFFLE): A
DROP SCORE: 0.0





  1%|          | 2/200 [00:19<31:04,  9.41s/it]


QUESTION: Which of these magazines does not focus on natural science?
CORRECT: A
MODEL (ORIG): A

PARAPHRASED: Which magazine among these does not emphasize natural science?
MODEL (PARA): A

STRUCTURAL REWRITE: Unlike other magazines, which focus on natural science, which one does not?
MODEL (STRUCT): D

DISTRACTOR MODIFIED OPTIONS:
A. Tiger Beat
B. Outside
C. National Geographic
D. Smithsonian
MODEL (DIST): A

SHUFFLED OPTIONS:
A. Smithsonian
B. Outside
C. National Geographic
D. Tiger Beat
CORRECT (SHUFFLED): D
MODEL (SHUFFLE): D
DROP SCORE: 0.25





  2%|▏         | 3/200 [00:26<27:21,  8.33s/it]

...


QUESTION:  Which is the main form of Buddhism in Sri Lanka?
CORRECT: A
MODEL (ORIG): A

PARAPHRASED: What is the dominant type of Buddhism practiced in Sri Lanka?
MODEL (PARA): A

STRUCTURAL REWRITE: Sri Lanka's predominant Buddhist form is Theravada.

Context:
 Buddhism has been the dominant religion in Sri Lanka for centuries. Out of the two major forms of Buddhism, Theravada and Mahayana, Theravada is the one
MODEL (STRUCT): A

DISTRACTOR MODIFIED OPTIONS:
A. Theravada
B. Mahayana
C. Pure Land
D. Zen
MODEL (DIST): A

SHUFFLED OPTIONS:
A. Theravada
B. Pure Land
C. Zen
D. Mahayana
CORRECT (SHUFFLED): A
MODEL (SHUFFLE): A
DROP SCORE: 0.0





100%|█████████▉| 199/200 [30:19<00:07,  7.51s/it]


QUESTION: Most surveillance systems use which of the following study designs?
CORRECT: B
MODEL (ORIG): B

PARAPHRASED: What research design is predominantly used by surveillance systems?
MODEL (PARA): D

STRUCTURAL REWRITE: What kind of study designs are commonly used in surveillance systems?
MODEL (STRUCT): D

DISTRACTOR MODIFIED OPTIONS:
A. Cohort
B. Serial cross-sectional
C. Mortality
D. Syndromic
MODEL (DIST): B

SHUFFLED OPTIONS:
A. Mortality
B. Syndromic
C. Cohort
D. Serial cross-sectional
CORRECT (SHUFFLED): D
MODEL (SHUFFLE): D
DROP SCORE: 0.5





100%|██████████| 200/200 [30:31<00:00,  9.16s/it]


QUESTION: Which of the following sequences of events is consistent with the presence of a layer of sandstone in bedrock just above a layer of shale?
CORRECT: B
MODEL (ORIG): B

PARAPHRASED: Can you determine which sequence of geological events corresponds to the existence of a sandstone layer over a shale layer in the rock strata?
MODEL (PARA): B

STRUCTURAL REWRITE: In order for there to be a layer of sandstone in bedrock just above a layer of shale, what sequence of events occurred?
MODEL (STRUCT): B

DISTRACTOR MODIFIED OPTIONS:
A. A volcano erupted, sending lava out over a layer of shale. The lava cooled and hardened into sandstone.
B. The water level of a large lake lowered. A beach then formed where previously there had been lake bottom.
C. One-celled organisms developed a colony on the seafloor. Shells made by these organisms accumulated and lithified, forming the sandstone.
D. Mud was deposited and lithified. Subsequent contact metamorphism resulted in localized recrystallizat

In [6]:
avg_drop = df["drop_score"].mean()
print("Average Perturbation Score:", avg_drop)

Average Perturbation Score: 0.1525


In [7]:
print("Original acc:",df["orig_correct"].mean())
print("Paraphrase acc:", df["paraphrase_correct"].mean())
print("Shuffle acc:", df["shuffle_correct"].mean())
print("Prefix acc:", df["structural_correct"].mean())
print("Distractor acc:", df["distractor_correct"].mean())

Original acc: 0.98
Paraphrase acc: 0.81
Shuffle acc: 0.71
Prefix acc: 0.82
Distractor acc: 0.98
